# Word Embeddings — Extended Practical Notebook

A step-by-step NLP lesson from sparse text features to dense word vectors. This notebook covers one-hot encoding, vocabulary indexes, embedding matrices, cosine similarity, sentence pooling, context windows, optional Word2Vec training with Gensim, and static versus contextual embeddings.

**Note:** handwritten vectors are illustrative values, not pretrained embeddings. The tiny Word2Vec corpus only demonstrates the API; it cannot produce reliable general-purpose vectors.

## Learning objectives

- Explain why text needs numerical representations.
- Compare one-hot, Bag of Words, TF-IDF, and dense embeddings.
- Understand embedding dimensions and matrix lookup.
- Calculate cosine similarity and find nearest neighbours.
- Build sentence vectors by averaging token vectors.
- Understand context windows and Word2Vec's CBOW/Skip-gram intuition.
- Distinguish static embeddings from contextual transformer representations.
- Recognize limitations, bias, and evaluation considerations.

## 1. Why convert words to numbers?

Models need numerical inputs. A token ID is only a label: if `cat=0` and `dog=1`, the value 1 does not mean dog is more similar to banana or cat. An embedding instead represents a token with a vector whose values are learned from data.

In [ ]:
words = ["cat", "dog", "banana"]
word_to_id = {word: index for index, word in enumerate(words)}
print(word_to_id)
print("IDs are lookup labels, not measurements of meaning.")

## 2. One-hot encoding

One-hot encoding assigns each vocabulary word a vector containing one 1 and otherwise 0s. It represents identity, not semantic similarity. With a vocabulary of size V, each one-hot vector has V entries.

In [ ]:
def one_hot(word, vocabulary):
    if word not in vocabulary:
        raise ValueError(f"{word!r} is not in the vocabulary")
    return [int(item == word) for item in vocabulary]

vocabulary = ["cat", "dog", "banana"]
for word in vocabulary:
    print(f"{word:>7} -> {one_hot(word, vocabulary)}")

In [ ]:
def dot_product(a, b):
    if len(a) != len(b):
        raise ValueError("Vectors must have equal length")
    return sum(x * y for x, y in zip(a, b))

print("cat · dog =", dot_product(one_hot("cat", vocabulary), one_hot("dog", vocabulary)))
print("cat · cat =", dot_product(one_hot("cat", vocabulary), one_hot("cat", vocabulary)))

## 3. Bag of Words and TF-IDF recap

Bag of Words counts tokens in a document. TF-IDF weights tokens using their frequency in a document and their rarity across documents. These are useful sparse baselines, but they do not usually learn a dense vector for each word.

In [ ]:
from collections import Counter
import re

documents = ["cats chase mice", "dogs chase cats", "ripe banana tastes sweet"]
def tokenize(text):
    return re.findall(r"[a-z]+", text.lower())

vocab = sorted({token for doc in documents for token in tokenize(doc)})
for doc in documents:
    counts = Counter(tokenize(doc))
    print(doc, "->", [counts[word] for word in vocab])
print("Vocabulary order:", vocab)

## 4. Dense word embeddings

A dense embedding maps each word to a compact vector, perhaps 50, 100, or 300 dimensions. Unlike one-hot vectors, many entries may be non-zero. In learned embeddings, meaning is distributed across dimensions; individual dimensions usually do not have simple, fixed human-readable labels.

In [ ]:
toy_embeddings = {
    "cat":    [0.80, 0.20, 0.10, 0.70],
    "kitten": [0.75, 0.25, 0.15, 0.72],
    "dog":    [0.70, 0.30, 0.20, 0.65],
    "banana": [0.10, 0.80, 0.90, 0.10],
}
for word, vector in toy_embeddings.items():
    print(f"{word:>7}: {vector} | dimension={len(vector)}")

## 5. Embedding matrix and token lookup

For vocabulary size V and embedding dimension d, an embedding matrix has shape `(V, d)`. Each row stores the vector for one token ID. The integer ID is used to retrieve the row; it does not itself carry meaning.

In [ ]:
vocabulary = ["banana", "cat", "dog", "kitten"]
word_to_index = {word: i for i, word in enumerate(vocabulary)}
index_to_word = {i: word for word, i in word_to_index.items()}
embedding_matrix = [
    [0.10, 0.80, 0.90],  # banana
    [0.80, 0.20, 0.10],  # cat
    [0.70, 0.30, 0.20],  # dog
    [0.75, 0.25, 0.15],  # kitten
]
word = "dog"
print("Word-to-index:", word_to_index)
print("ID:", word_to_index[word])
print("Embedding lookup:", embedding_matrix[word_to_index[word]])
print("Matrix shape:", (len(embedding_matrix), len(embedding_matrix[0])))

## 6. Vector arithmetic

Vectors can be added, subtracted, and scaled. Such operations are mathematically straightforward, but a semantic interpretation only makes sense when vectors were trained appropriately. Do not infer linguistic rules from the handwritten values in this notebook.

In [ ]:
def vector_add(a, b):
    if len(a) != len(b):
        raise ValueError("Vectors must have equal length")
    return [x + y for x, y in zip(a, b)]

def vector_subtract(a, b):
    if len(a) != len(b):
        raise ValueError("Vectors must have equal length")
    return [x - y for x, y in zip(a, b)]

print(vector_add([1, 2, 3], [0.5, 1, 1.5]))
print(vector_subtract([1, 2, 3], [0.5, 1, 1.5]))

## 7. Cosine similarity

For non-zero vectors a and b, cosine similarity is the dot product divided by the product of their lengths. Similar directions yield scores near 1, perpendicular directions near 0, and opposite directions near -1. Interpret scores relative to the embedding model and task.

In [ ]:
from math import sqrt

def cosine_similarity(a, b):
    if len(a) != len(b):
        raise ValueError("Vectors must have equal length")
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = sqrt(sum(x * x for x in a))
    norm_b = sqrt(sum(y * y for y in b))
    if norm_a == 0 or norm_b == 0:
        raise ValueError("Cosine similarity is undefined for zero vectors")
    return dot / (norm_a * norm_b)

print("Same direction:", cosine_similarity([1, 2], [2, 4]))
print("Perpendicular:", cosine_similarity([1, 0], [0, 1]))
print("Opposite direction:", cosine_similarity([1, 0], [-1, 0]))

In [ ]:
for first, second in [("cat", "kitten"), ("cat", "dog"), ("cat", "banana")]:
    score = cosine_similarity(toy_embeddings[first], toy_embeddings[second])
    print(f"{first:>6} vs {second:<7}: {score:.4f}")

## 8. Find nearest neighbours

Nearest-neighbour search ranks candidate vectors by similarity. Large-scale systems often use specialized vector search indexes; this small function is enough for a basic example.

In [ ]:
def most_similar(query_word, vectors, top_n=3):
    if query_word not in vectors:
        raise KeyError(f"Unknown word: {query_word}")
    scores = [
        (word, cosine_similarity(vectors[query_word], vector))
        for word, vector in vectors.items()
        if word != query_word
    ]
    return sorted(scores, key=lambda item: item[1], reverse=True)[:top_n]

for word, score in most_similar("cat", toy_embeddings):
    print(f"{word:>7}: {score:.4f}")

## 9. Sentence vectors using mean pooling

A simple sentence representation is the average of its word vectors. It produces a fixed-size vector for sentences of different lengths. However, averaging loses word order and may blur negation or different word senses.

In [ ]:
def mean_pool(words, vectors):
    known = [vectors[word] for word in words if word in vectors]
    if not known:
        return None
    dimensions = len(known[0])
    return [sum(vector[i] for vector in known) / len(known)
            for i in range(dimensions)]

s1 = mean_pool(["cat", "kitten"], toy_embeddings)
s2 = mean_pool(["cat", "dog"], toy_embeddings)
s3 = mean_pool(["banana"], toy_embeddings)
print("Sentence 1:", s1)
print("Sentence 2:", s2)
print("Sentence 3:", s3)
print("Similarity s1/s2:", cosine_similarity(s1, s2))
print("Similarity s1/s3:", cosine_similarity(s1, s3))

### Weighted averaging

A weighted average gives some tokens more influence than others. TF-IDF weights are one possible baseline. Weighted averaging is still not the same as a dedicated, trained sentence embedding model.

In [ ]:
def weighted_pool(words, vectors, weights):
    selected = [(vectors[word], weights.get(word, 1.0))
                for word in words if word in vectors]
    if not selected:
        return None
    total = sum(weight for _, weight in selected)
    if total == 0:
        raise ValueError("Total weight must not be zero")
    dimensions = len(selected[0][0])
    return [sum(vec[i] * weight for vec, weight in selected) / total
            for i in range(dimensions)]

weights = {"cat": 2.0, "kitten": 1.5, "dog": 1.0}
print(weighted_pool(["cat", "kitten", "dog"], toy_embeddings, weights))

## 10. Context windows

Embedding methods use context patterns. A context window defines how many nearby tokens are considered around a target word. The example below generates target-context pairs within a symmetric window.

In [ ]:
def context_pairs(tokens, window_size=2):
    pairs = []
    for i, target in enumerate(tokens):
        start = max(0, i - window_size)
        end = min(len(tokens), i + window_size + 1)
        for j in range(start, end):
            if i != j:
                pairs.append((target, tokens[j]))
    return pairs

for pair in context_pairs(["cats", "chase", "small", "mice"], window_size=2):
    print(pair)

## 11. Optional: train Word2Vec with Gensim

Install Gensim in the same Python environment as your notebook kernel:

```bash
python -m pip install gensim
```

If the import fails, check the active interpreter with `import sys; print(sys.executable)`. The tiny sample corpus below only demonstrates the API. Its learned vectors are not reliable general-purpose embeddings.

In [ ]:
raw_sentences = [
    "cats chase small mice",
    "dogs chase small cats",
    "mice hide in small spaces",
    "dogs and cats are animals",
    "kittens are young cats",
    "puppies are young dogs",
    "cats and dogs can be pets",
    "small animals need food",
]
tokenized_sentences = [tokenize(sentence) for sentence in raw_sentences]
for sentence in tokenized_sentences:
    print(sentence)

In [ ]:
try:
    from gensim.models import Word2Vec
    gensim_available = True
    print("Gensim is available in this kernel.")
except ImportError:
    gensim_available = False
    print("Gensim is not installed in this kernel.")
    print("Install it with: python -m pip install gensim")

In [ ]:
if gensim_available:
    w2v_model = Word2Vec(
        sentences=tokenized_sentences,
        vector_size=30,
        window=2,
        min_count=1,
        workers=1,
        sg=1,       # 1 = Skip-gram, 0 = CBOW
        seed=42,
        epochs=100,
    )
    print("Vocabulary size:", len(w2v_model.wv))
    print("Vector shape:", w2v_model.wv["cats"].shape)
    print("First ten vector values:", w2v_model.wv["cats"][:10])
else:
    print("Skip training until Gensim is installed.")

In [ ]:
if gensim_available:
    print("Nearest words to cats:")
    for word, score in w2v_model.wv.most_similar("cats", topn=5):
        print(f"{word:>10}: {score:.4f}")
else:
    print("No trained model available in this session.")

In [ ]:
if gensim_available:
    for first, second in [("cats", "dogs"), ("cats", "kittens"), ("cats", "banana")]:
        if first in w2v_model.wv and second in w2v_model.wv:
            print(first, "vs", second, round(w2v_model.wv.similarity(first, second), 4))
else:
    print("Install Gensim to compare learned vectors.")

### CBOW vs Skip-gram

- **CBOW:** predicts a target word from surrounding context words.
- **Skip-gram:** predicts surrounding context words from a target word.

In Gensim's `Word2Vec`, `sg=0` selects CBOW and `sg=1` selects Skip-gram. The separate [Word2Vec lesson](../05_Word2Vec/README.md) explores these models in more depth.

In [ ]:
if gensim_available:
    cbow_model = Word2Vec(
        sentences=tokenized_sentences,
        vector_size=30,
        window=2,
        min_count=1,
        workers=1,
        sg=0,
        seed=42,
        epochs=100,
    )
    print("CBOW vector shape:", cbow_model.wv["cats"].shape)
else:
    print("Gensim is required for the CBOW demonstration.")

## 12. Static vs contextual embeddings

Word2Vec and GloVe usually provide a single vector per vocabulary word. This is called a static embedding. The word `bank` therefore has one vector in both financial and river contexts.

Transformer models use surrounding tokens to build contextual representations. The representation of `bank` can differ between “I deposited money at the bank” and “We sat on the river bank.” The exact embedding depends on the model, tokenizer, layer, and pooling method.

In [ ]:
examples = [
    "I deposited money at the bank.",
    "We sat on the river bank.",
]
for sentence in examples:
    print(sentence)
print("\nStatic embedding: one stored vector for bank.")
print("Contextual embedding: representation can vary with sentence context.")

## 13. Compare representation methods

| Method | Representation | Advantage | Limitation |
|---|---|---|---|
| One-hot | Sparse token vector | Simple identity representation | No semantic closeness |
| Bag of Words | Sparse document counts | Interpretable baseline | Usually ignores word order |
| TF-IDF | Sparse weighted terms | Highlights informative words | Limited semantic generalization |
| Word2Vec | Static dense word vector | Learns usage patterns | One vector per word |
| GloVe | Static dense word vector | Uses global co-occurrence | One vector per word |
| Transformer | Contextual token vectors | Context-sensitive representation | More computationally expensive |
| Sentence embedding model | Whole-sentence vector | Useful for semantic search | Quality depends on model/task |

## 14. Applications

Embeddings are used for semantic search, retrieval, text classification, clustering, duplicate-question detection, recommendation, document matching, and as inputs to neural NLP systems. In retrieval-augmented generation, an embedding model can help find relevant documents before a language model generates a response.

An embedding alone is not a complete system: the dataset, downstream model, evaluation metric, and deployment context all matter.

## 15. Limitations and evaluation

- **Data dependence:** vectors reflect patterns in the training corpus.
- **Bias:** social and cultural biases can be encoded in representations.
- **Vocabulary coverage:** older static models may not represent unseen words well.
- **Polysemy:** one static vector can mix multiple meanings.
- **Domain shift:** vectors learned on one domain may perform poorly on another.
- **Similarity is not truth:** related words, synonyms, and opposites can all occur in similar contexts.
- **Tiny corpus warning:** toy data is not sufficient for production quality.

For a real project, define the task, compare against a TF-IDF baseline, evaluate on held-out data, inspect nearest neighbours, check bias, and record preprocessing and model versions.

## 16. Summary and references

Remember: embeddings are dense vectors learned from data; token IDs are lookup indices; cosine similarity compares vector directions; averaging word vectors loses order; Word2Vec/GloVe are usually static; transformer representations are contextual.

References:
- [Word2Vec paper](https://arxiv.org/abs/1301.3781)
- [GloVe project](https://nlp.stanford.edu/projects/glove/)
- [Gensim Word2Vec documentation](https://radimrehurek.com/gensim/models/word2vec.html)
- [Attention Is All You Need](https://arxiv.org/abs/1706.03762)
- [scikit-learn cosine similarity](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html)